# OUMUAMUA - En búsqueda de su estrella madre

## Librerías

In [1]:
%pip install pymcel celluloid astroquery astropy scipy numpy pandas matplotlib -Uq

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.4 which is incompatible.
uxarray 2025.6.0 requires numpy<2.3, but you have numpy 2.4.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!



## Estado inicial de referencia (base común)
Extraer un estado heliocéntrico de 1I/'Oumuamua cercano a perihelio y utilizarlo como condición de referencia para todos los experimentos. Según wikipedia, esto representa la época del 14 de octubre de 2017.

In [41]:
epoch_ref = '2017-10-14'  # Fecha de referencia cercana al perihelio

def extrae_estado(resultado):
    if isinstance(resultado, tuple) and len(resultado) >= 3:
        estado = np.asarray(resultado[2], dtype=float)
        if estado.ndim == 2:
            estado = estado[0]
        return estado
    return np.asarray(resultado, dtype=float)

raw = pc.consulta_horizons(
    id='1I',
    location='@0',
    epochs=epoch_ref,
    datos='vectors',
    propiedades=[('x', 'km'), ('y', 'km'), ('z', 'km'), ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')]
)

estado_ref = extrae_estado(raw)
r0, v0 = estado_ref[:3], estado_ref[3:]

print('Estado extraído correctamente:')
print('r0 [km]:', r0)
print('v0 [km/s]:', v0)


Estado extraído correctamente:
r0 [km]: [ 1.45267492e+08  7.47620204e+07 -1.07128187e+07]
v0 [km/s]: [44.83537051 10.39892688 14.33813591]


## Caracterización de la hipérbola
Se obtienen elementos orbitales con `estado_a_elementos(mu, estado)` y se valida energía específica positiva.

In [42]:
mu_sun = pc.constantes.mu_sun  # valor base en SI (m^3/s^2)
mu_km = mu_sun / 1e9           # conversión a km^3/s^2 (consistente con r0, v0)

print('Parámetro gravitacional del Sol [m^3/s^2]:', mu_sun)
print('Parámetro gravitacional del Sol [km^3/s^2]:', mu_km)

# elementos = (p, e, i, Omega, omega, f) según convención de pymcel.
p, e, inc, Omega, omega, f = pc.estado_a_elementos(mu_km, estado_ref)

# Validación física rápida para clasificar la cónica
energia_especifica = 0.5 * np.dot(v0, v0) - mu_km / np.linalg.norm(r0)

print('Elementos orbitales extraídos correctamente:')
print('p [km]:', p)
print('e:', e)
print('i [°]:', np.degrees(inc))
print('Omega [°]:', np.degrees(Omega))
print('omega [°]:', np.degrees(omega))
print('f [°]:', np.degrees(f))
print('Energía específica [km^2/s^2]:', energia_especifica)
print('¿Órbita hiperbólica (ε>0)?', energia_especifica > 0)


Parámetro gravitacional del Sol [m^3/s^2]: 1.3271244004127942e+20
Parámetro gravitacional del Sol [km^3/s^2]: 132712440041.27942
Elementos orbitales extraídos correctamente:
p [km]: 85604594.26472798
e: 1.20554107345583
i [°]: 123.1137532641553
Omega [°]: 24.781488427458203
omega [°]: 242.20374636009197
f [°]: 113.31585543976549
Energía específica [km^2/s^2]: 351.3972315370671
¿Órbita hiperbólica (ε>0)? True


En dos cuerpos, la energía específica y el momento angular específico son invariantes.

In [43]:
def invariantes_dos_cuerpos(mu, r, v):
    eps = 0.5 * np.sum(v * v, axis=1) - mu / np.linalg.norm(r, axis=1)
    h = np.cross(r, v)
    hnorm = np.linalg.norm(h, axis=1)
    return eps, hnorm

# convertir mu a km^3/s^2 y asegurar r,v con dimensión (N,3)
mu_km = mu_sun / 1e9
ere, mare = invariantes_dos_cuerpos(mu_km, np.atleast_2d(r0), np.atleast_2d(v0))
ere = ere[0]
mare = mare[0]
print('Invariantes de dos cuerpos calculadas correctamente:')  
print('Energía relativa específica (ERE) [km^2/s^2]:', ere)
print('Magnitud del momento angular específico (MARE) [km^2/s]:', mare)

Invariantes de dos cuerpos calculadas correctamente:
Energía relativa específica (ERE) [km^2/s^2]: 351.3972315370671
Magnitud del momento angular específico (MARE) [km^2/s]: 3370577781.8670444


In [33]:
def tipo_orbita_conica(energia_especifica, mu, tolerancia=1e-6):
    """
    Determina el tipo de órbita cónica según la energía específica.
    Retorna:
    --------
    str : Tipo de órbita ('Elíptica', 'Parabólica', 'Hiperbólica')
    """
    if energia_especifica < -tolerancia:
        return 'Elíptica'
    elif abs(energia_especifica) < tolerancia:
        return 'Parabólica'
    else:
        return 'Hiperbólica'

# Evaluar con nuestro ere
tipo = tipo_orbita_conica(ere, mu_km)
print(f'Tipo de órbita de 1I/Oumuamua: {tipo}')
print(f'Energía específica ε: {ere:.6f} km²/s²')

Tipo de órbita de 1I/Oumuamua: Hiperbólica
Energía específica ε: 350.538258 km²/s²
